# Paper combined - train the edit model, then run BCO experiments

**PART 1** trains the edit agent FROM SCRATCH on the 700-graph graded realistic curriculum (dup -> stub -> detour -> drop_cover -> subtle -> eval-mix -> clean) with the unified objective: 0.5*RTT + 0.5*WMC and a network-level adjustment budget (cap at target=0.2, W=10, paper mode, demand off; the BCO search uses the two-sided |adj-target| form). **PART 2** runs the paper experiments (E1-E6) using that fine-tuned model as `OUR_MODEL_PATH`. Algorithm budgets are set for a full top-to-bottom run that should fit into roughly 25-30 hours on the current CPU calibration; rerun the timing estimator cell after any budget change.

Run top-to-bottom: training must finish before the experiment config picks up the trained weights.


In [1]:
import torch

torch.__version__

'2.11.0+cpu'

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # select GPU 1
import sys, json, random as _random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

from connectpt.routes_generator.core.paths import (ROOT_DIR, CFG_DIR, DATASETS_DIR,
                                                   MODEL_OUTPUTS_DIR, EDIT_MODEL_WEIGHTS_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator.improvement_learning import load_raw_graphs_and_lc_routes
from connectpt.routes_generator.bee_colony import get_adjustment_degrees
from connectpt.routes_generator.reports.results_io import save_table
from connectpt.routes_generator.lc_eval import build_lc_cfg, run_lc
from connectpt.routes_generator.data import as_route_tensor
from connectpt.routes_generator.reports import route_plots

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from connectpt.routes_generator.core.paths import ROOT_DIR, EDIT_MODEL_WEIGHTS_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
# Library names the PART-2 cells use (baselines runners/builders + data helpers).
from connectpt.routes_generator.data.loaders import BENCHMARK_SPECS
from connectpt.routes_generator.data import as_route_tensor
from connectpt.routes_generator.baselines import load_benchmark_graph, _run_baseline
from connectpt.routes_generator.improvement_learning import load_raw_graphs_and_lc_routes
from connectpt.routes_generator.reports import route_plots
from connectpt.routes_generator.bee_colony import get_adjustment_degrees

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.max_columns", None)
print("device:", device)

# --- wall-clock instrumentation (per-epoch / per-iteration) ---
import time as _time
TIMING = {"epoch_s": None, "dataset_graph_s": None, "bco_iter_s": {}, "base_iter_s": {}}

# --- experiment-suite profile: ONE source of truth = cfg/experiments/suite*.yaml ---
# 'suite_smoke' = every experiment ON, single city, TEMP_ outputs (a dry-run that
# never overwrites real files); 'suite' = the full paper run.
from connectpt.routes_generator.core import load_suite
from connectpt.routes_generator.suite_context import RunContext
SUITE = load_suite("suite_smoke")
# Explicit runtime knobs (edit checkpoint, init mode, output prefix) -- one
# immutable object passed to runners/sinks instead of module globals.
CTX = RunContext.from_suite(SUITE)
print(f"suite profile={SUITE.profile} smoke={SUITE.smoke} prefix={CTX.output_prefix!r} "
      f"cities={list(SUITE.cities)}")

## Configuration

`copy_full`, `copy_boundary`, `copy_mixed`, and `lc_clean` are equal-size
tiers.  The three corrupted tiers are generated from LC routes with the four
copy/subcopy mutations below.  Multiplicity grows across the curriculum up to
five routes on one stop-to-stop leg.

In [ ]:
# --- training configuration: cfg/train/edit_scratch[_smoke].yaml, selected by the
# suite profile loaded above (smoke -> tiny dataset, 1 iteration, TEMP_ outputs). ---
from connectpt.routes_generator.paper_experiments.training_lc import load_train_config
train_cfg = load_train_config("edit_scratch_smoke" if SUITE.smoke else "edit_scratch")
print(f"train cfg: run={train_cfg.run.name}, n_iterations={train_cfg.train_loop.n_iterations}, "
      f"dataset={train_cfg.data.dataset_dirname}, tiers={list(train_cfg.curriculum.tiers)}")


## Generate the four-tier route-copy dataset

Every corruption copies a whole donor route or a contiguous donor subroute
into another route slot.  Prefix, suffix, and interior replacements preserve
the recipient length.  Candidates with repeated stops are rejected, so the
dataset teaches inter-route redundancy rather than synthetic self-loops.

In [ ]:
if SUITE.run.training:
    # Dataset generation: build the copy-tier dataset straight from the cfg
    # (experiments.training_lc.copytier_config reads cfg.dataset_gen / data /
    # curriculum; no N_GRAPHS / RAW_* / LC_COMBOS constants in the notebook).
    from connectpt.routes_generator.paper_experiments.training_lc import copytier_config, build_copytier_dataset

    seconds = build_copytier_dataset(copytier_config(train_cfg))
    if seconds is not None:
        TIMING["dataset_graph_s"] = seconds
        print(f"[timing] dataset gen: {seconds:.2f} s/graph "
              f"(N_GRAPHS={train_cfg.dataset_gen.n_graphs})")


## Load, split, and define the cumulative curriculum

In [ ]:
if SUITE.run.training:
    # Load graphs + the tier-stratified split for the dormant report cells.
    # Geometry/paths come from copytier_config(train_cfg); the split reproduces
    # TrainingDataModule.stratified_split (same logic EditTrainingRun uses).
    from connectpt.routes_generator.training import TrainingDataModule
    from connectpt.routes_generator.paper_experiments.training_lc import copytier_config

    dataset_cfg = copytier_config(train_cfg)
    data_module = TrainingDataModule(
        raw_graphs_path=dataset_cfg.subset_pkl, lc_results_dir=dataset_cfg.new_dataset_dir, device=device,
        min_route_len=dataset_cfg.min_route_len, max_route_len=dataset_cfg.max_route_len,
        target_n_routes=dataset_cfg.target_n_routes).setup()
    graphs, seed_routes = data_module.graphs, data_module.seed_routes
    meta_df = pd.read_csv(dataset_cfg.meta_csv)
    print(f"loaded {len(graphs)} graphs; seed_routes={tuple(seed_routes.shape)}")
    display(meta_df.groupby("tier")[[
        "applied_events", "redun_before", "redun_after",
        "max_leg_use_after", "d_un_after_pct"]].mean().round(3).reindex(dataset_cfg.tiers))

    tier_of = dict(zip(meta_df["graph_index"], meta_df["tier"]))
    (TRAIN_INDICES, VAL_INDICES, MONITOR_VAL_INDICES,
     train_by_tier, val_by_tier) = data_module.stratified_split(
        tier_of, dataset_cfg.tiers, train_fraction=float(train_cfg.data.train_fraction),
        n_val_per_tier=int(train_cfg.curriculum.n_val_per_tier),
        seed=int(train_cfg.data.split_seed))
    print(f"validation graphs={len(VAL_INDICES)}; monitor={len(MONITOR_VAL_INDICES)}")

    # curriculum stage spans (start,end,label) for the history-figure shading
    n_iter = int(train_cfg.train_loop.n_iterations)
    schedule = [(round(float(f) * n_iter), list(t), lab)
                 for f, t, lab in train_cfg.curriculum.schedule]

    def stage_spans():
        spans, prev = [], 0
        for until, _t, label in schedule:
            spans.append((prev + 1, until, label)); prev = until
        return spans


## Model and objective builders

Two helpers used by the fine-tune run below. `build_edit_run` composes the
hydra config, builds a **fresh** trim model + cost module, selects which of
the three cost components (`demand` / `route` / `connectivity`) are active,
and enables the optional adjustment-degree shaping through `adj_weight`.
`train_edit_run` wraps `train_lc_improvement_cfg` and returns the in-memory
history. Adjustment conditioning stays off, so the fine-tuned actor keeps the
same architecture as the RTT+connectivity checkpoint.


In [ ]:
# Config-first model/cost builder for the dormant standalone balanced-eval cell.
# There is no bespoke "build_edit_run" anymore -- the real training runs via
# EditTrainingRun (cell 15). This re-exports the library helpers that build the
# edit model + unified cost config-first from train/edit + the objective YAML;
# the old 60-line builder (with vestigial adj_*/critic params it never used) and
# the rollout-kwargs helper now live in experiments/training_lc.py.
from connectpt.routes_generator.paper_experiments.training_lc import build_edit_model_and_cost, rollout_adjustment_kwargs

print("builders ready: build_edit_model_and_cost(), rollout_adjustment_kwargs() "
      "[config-first via experiments.training_lc]")


## Clean LC baseline costs for history plots

Generate clean LC routes for the validation monitor graphs, score them with the same route/connectivity scalar used by the training-history plots, and save the result as a standalone CSV baseline.


In [ ]:
# Clean-LC baseline for the training-history figure. Knobs live in
# cfg.report.baseline; heavy logic in experiments.training_lc.clean_lc_baseline.
if SUITE.run.clean_lc_baseline:
    from connectpt.routes_generator.paper_experiments.training_lc import clean_lc_baseline, copytier_config
    dataset_cfg = copytier_config(train_cfg)
    if "graphs" not in globals() or "seed_routes" not in globals():
        graphs, seed_routes = load_raw_graphs_and_lc_routes(dataset_cfg.subset_pkl, dataset_cfg.new_dataset_dir)
    if "meta_df" not in globals():
        meta_df = pd.read_csv(dataset_cfg.meta_csv)
    clean_lc_baseline_df = clean_lc_baseline(
        train_cfg, graphs=graphs, seed_routes=seed_routes, meta_df=meta_df, device=device,
        baseline_path=MODEL_OUTPUTS_DIR / f"{train_cfg.run.name}_clean_lc_baseline_cost.csv")
    display(clean_lc_baseline_df.round(4))


## From-scratch training - route + connectivity + adj (W=10)

This run starts from the final-experiments RTT + connectivity checkpoint,
keeps `route` and `connectivity` active at 0.5/0.5 (demand off), and adds
network-level adjustment-budget reward shaping (cap at target=0.2, W=10, paper mode). The budget is 100
epochs over the current 500-graph copy-redundancy curriculum, batch 4, with
balanced per-tier validation monitoring. The resulting checkpoint is saved
under a new `adjcap` run name and is used by the evaluation cells below.

If `RESUME_FROM_CHECKPOINT=True`, this cell reloads the adj-finetune output
first; otherwise it initializes from `BASE_MODEL_PATH`.


In [ ]:
if SUITE.run.training:
    # Config-first training: EditTrainingRun builds model + cost + curriculum +
    # data from train_cfg (cfg/train/edit_scratch.yaml) and trains it.
    from pathlib import Path
    from connectpt.routes_generator.training import EditTrainingRun

    artifact = EditTrainingRun(train_cfg).run()
    history_df = artifact.history
    BEST_MODEL_PATH = Path(artifact.checkpoint_path)
    print(f"training done -> {BEST_MODEL_PATH} ({len(history_df)} epochs)")


## Train the edit model (cumulative curriculum)

Fine-tune the edit model over the cumulative curriculum (duplicate -> boundary
-> mixed -> covered -> clean). Training history is checkpointed to a partial CSV;
on resume the prior history is prepended.

In [ ]:
if SUITE.run.training:
    # History stitching + actor curves + TensorBoard mirroring (cfg-driven:
    # run name + scalar allow-list come from train_cfg.report).
    from connectpt.routes_generator.paper_experiments.training_lc import stitch_history, plot_training_history

    h, spans = stitch_history(
        prior_history_files=(),
        history_df=globals().get("history_df"),
        full_history_checkpoint=MODEL_OUTPUTS_DIR / f"{train_cfg.run.name}_training_history_partial.csv")
    if not spans and "stage_spans" in globals():
        spans = stage_spans()
    shade = plot_training_history(h, spans, train_cfg, model_outputs_dir=MODEL_OUTPUTS_DIR)
    print(f'[tensorboard] launch:  tensorboard --logdir "{MODEL_OUTPUTS_DIR / "tensorboard"}"')


## Critic diagnostics (per-component critic MSE / explained variance)

In [ ]:
if SUITE.run.training:
    from connectpt.routes_generator.paper_experiments.training_lc import plot_critic_metrics
    crit_table = plot_critic_metrics(h, shade)
    if crit_table is not None:
        display(crit_table)


## Evaluate balanced policy by tier

The table reports before/after route-level redundancy, `ATT`, `RTT`,
connectivity, and demand percentages by transfer bucket (`d0`, `d1`, `d2`,
`d_un`). `Adj(current, seed)` remains available only in the example plots.

In [ ]:
if SUITE.run.training:
    # Standalone balanced eval: rebuild model+cost from the checkpoint if the
    # training cell wasn't run, then evaluate per tier (cfg-driven).
    from connectpt.routes_generator.paper_experiments.training_lc import balanced_eval_by_tier

    if "model" not in globals() or "cost_obj" not in globals():
        _, cost_obj, model, _, BEST_MODEL_PATH = build_edit_model_and_cost(
            run_name=train_cfg.run.name, device=device,
            vary_weights=bool(train_cfg.cost.variable_weights),
            adj_weight=float(train_cfg.adjustment_degree_weight))
        if not BEST_MODEL_PATH.exists():
            raise FileNotFoundError(
                f"No trained checkpoint at {BEST_MODEL_PATH}. Train first, or check run.name.")
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
        model.eval()
        print(f"[eval] loaded trained model <- {BEST_MODEL_PATH.name} (training cell not run)")

    eval_df, visual_examples = balanced_eval_by_tier(
        train_cfg, model=model, cost_obj=cost_obj, device=device,
        graphs=graphs, seed_routes=seed_routes, val_by_tier=val_by_tier)
    display(eval_df)
    save_table(eval_df, f"{train_cfg.run.name}_eval_by_tier", prefix=CTX.output_prefix)
    vx_path = MODEL_OUTPUTS_DIR / f"{train_cfg.run.name}_visual_examples.pt"
    torch.save(visual_examples, vx_path)
    print(f"[eval] saved visual_examples -> {vx_path}")
    print("ATT/RTT/CONN are minutes; d0/d1/d2/d_un are demand percentages by transfer bucket.")


## Visual validation examples

For the balanced preference vector, show one seed and the corresponding edited
network from every tier.  The right-hand panels emphasize removed and added
segments relative to the corrupted seed.

In [ ]:
if SUITE.run.training:
    from connectpt.routes_generator.paper_experiments.training_lc import plot_balanced_examples
    if "visual_examples" not in globals() or not visual_examples:
        vx_path = MODEL_OUTPUTS_DIR / f"{train_cfg.run.name}_visual_examples.pt"
        if vx_path.exists():
            try:
                visual_examples = torch.load(vx_path, map_location="cpu", weights_only=False)
            except TypeError:
                visual_examples = torch.load(vx_path, map_location="cpu")
            print(f"[viz] loaded visual_examples <- {vx_path.name}")
        else:
            visual_examples = {}
    plot_balanced_examples(visual_examples, graphs, list(train_cfg.curriculum.tiers))


## Post-training convergence: RPC + type2 vs RPC + trim/extend (Mumford1, 100 iters)

Right after PART 1 training, run two of the 5-model BCO variants -- random
path-combiner (RPC) rebuild paired with the hand-designed `type2` edit vs the
freshly trained `trim/extend` edit bee -- on Mumford1 for 100 BCO iterations
(balanced alpha=0.5, adjustment penalty off) and plot the convergence history.

In [ ]:
if SUITE.run.training:
    # Post-PART-1 convergence: two 5-model BCO variants driven by the just-trained
    # edit checkpoint (city/iters/alpha from train_cfg.report.post_train). BCO
    # logic in training_lc; the notebook keeps only the convergence plot.
    import matplotlib.pyplot as plt
    from connectpt.routes_generator.paper_experiments.training_lc import post_training_convergence

    conv_df, curves = post_training_convergence(
        train_cfg, best_model_path=BEST_MODEL_PATH,
        benchmark_specs=BENCHMARK_SPECS, load_benchmark_graph=load_benchmark_graph)
    display(conv_df)

    if curves:
        post = train_cfg.report.post_train
        fig, ax = plt.subplots(figsize=(8, 5))
        for label, curve in curves.items():
            ax.plot(range(1, len(curve) + 1), curve, marker="o", ms=2, label=label)
        ax.set_xlabel("BCO iteration")
        ax.set_ylabel("best objective cost (lower = better)")
        ax.set_title(f"Post-training convergence -- {post.city} "
                     f"(alpha={post.alpha}, adj off, {post.iters} iters)")
        ax.grid(alpha=0.3); ax.legend()
        plt.show(); plt.close(fig)
        curves_df = pd.DataFrame({k: pd.Series(v) for k, v in curves.items()})
        curves_df.index.name = "bco_iteration"
        save_table(curves_df.reset_index(), f"posttrain_convergence_{post.city}",
                   prefix=CTX.output_prefix)
    else:
        print("[post-train] no convergence history captured")


---
# PART 2 - BCO experiments (E1-E6)

Uses the model trained above (`BEST_MODEL_PATH`). All baselines are re-run live on a single seed with small budgets.

## Configuration

`SMOKE=True` -> Mandl only, 1 seed, tiny iteration budgets (just checks the
pipeline builds tables/figures). `SMOKE=False` runs Mandl and Mumford0-3
with the 25-30 hour budgeted settings below.


In [ ]:
# --- PART 2 experiment runtime setup (driven by the suite profile, SUITE) ---
# All experiment SELECTION + parameters live in cfg/experiments/suite*.yaml (read
# as SUITE in the setup cell). Runtime knobs (edit checkpoint, output prefix,
# benchmark init mode) travel via CTX (RunContext) -- no module globals.
OUR_MODEL_PATH = CTX.edit_weights_path

# Per-city algorithm budgets (SA/HH/GA/BCO iteration counts, populations, the
# Mandl BCO temperature schedule) live in cfg/search/budgets.yaml.
from connectpt.routes_generator.budgets import make_algo_settings, load_budgets
_BUDGETS = load_budgets()
HH_MAX_REPAIR_ITERS = _BUDGETS["hh_max_repair_iters_default"]
E2_BCO_ITERATIONS = _BUDGETS["e2_bco_iterations"]
algo_settings = make_algo_settings(SUITE.smoke, SUITE.quick, budgets=_BUDGETS)

print("cities:", list(SUITE.cities), "| our model:", OUR_MODEL_PATH.name,
      "| exists:", OUR_MODEL_PATH.exists())
print("output prefix:", repr(SUITE.output_prefix), "| enabled experiments:", dict(SUITE.run))
print("E1 narrow:", list(SUITE.e1.cities), "| alphas", list(SUITE.e1.alpha_grid),
      "| target", SUITE.e1.adj_target, "| E2 BCO iters:", E2_BCO_ITERATIONS)


## Shared PART-2 imports (paper sinks + metric helpers)

In [ ]:
# Shared PART-2 imports: paper_results sinks + the metric helpers the experiment
# cells below use (library reports/evaluation layer). Config composition is
# config-first (cfg/experiments/* declarative run/batch configs); init networks
# come from the data sources (connectpt.routes_generator.data) -- no experiment
# logic lives inline.
from tqdm.auto import tqdm

from connectpt.routes_generator.reports.paper_io import (PAPER_DIR, paper_row as _row,
                                                        save_paper_routes, save_paper_table)
from connectpt.routes_generator.evaluation import adj_vs_init

## EKB case study

Load the Ekaterinburg instance, inspect its projected coordinates, and render the supplied seed routes as a regular matplotlib figure.


In [ ]:
# EKB case study: load the instance + render its seed route network (geo/GIS).
import pandas as pd
from types import SimpleNamespace
from IPython.display import display
from connectpt.routes_generator.data import EKBDataSource
from connectpt.routes_generator.reports import (render_report, network_connectivity_stats,
                                                route_stats, project_coords)
from connectpt.routes_generator.reports.ekb import EKB_COORD_CRS

if SUITE.run.ekb_case_study:
    inst = EKBDataSource().load()
    _stats = route_stats(inst.init_routes)
    _latlon = project_coords(inst.coords, EKB_COORD_CRS)
    print("EKB coords:", tuple(inst.coords.shape), "CRS", EKB_COORD_CRS,
          "lat", (round(float(_latlon[:, 0].min()), 4), round(float(_latlon[:, 0].max()), 4)),
          "lon", (round(float(_latlon[:, 1].min()), 4), round(float(_latlon[:, 1].max()), 4)))
    _conn = network_connectivity_stats(inst.tensors, inst.init_routes)
    print("EKB spec:", inst.spec, "| route stats:", _stats)
    print("EKB connectivity:", {"components": _conn["n_components"],
          "isolated_nodes": _conn["isolated_nodes"], "symmetric": _conn["symmetric"],
          "cross_component_demand_pct": round(_conn["cross_component_demand_pct"], 4)})
    _art = SimpleNamespace(run_name="EKB seed", table=pd.DataFrame(),
                           routes={"Initial EKB routes": inst.init_routes},
                           instance=inst, metadata={})
    for fig in render_report(_art, kind="gis", title="EKB seed").figures.values():
        display(fig)
else:
    print("[EKB case study] skipped (SUITE.run.ekb_case_study=False)")

### EKB NBCO GNN + trim/extend

Run our NBCO variant on the Ekaterinburg instance and draw before/after route sets in both plain and diff modes. Overlapping route edges are rendered as curved arcs so duplicate coverage stays visible instead of collapsing into one line.


In [ ]:
# EKB NBCO (GNN rebuild + trim/extend) via the top C-API + geo/GIS render.
# Config-first: cfg/experiments/ekb/case_study/our_nbco[_smoke].yaml (seeded
# alpha sweep on the EKB network, sequential bees). render_report(kind="gis")
# draws the geo route panels + a diff-vs-Initial grid over the street underlay.
from IPython.display import display
from connectpt.routes_generator import ExperimentRunFactory, load_experiment, render_report

if SUITE.run.ekb_nbco:
    _name = "ekb/case_study/our_nbco" + ("_smoke" if SUITE.smoke else "")
    EKB_ART = ExperimentRunFactory.from_cfg(load_experiment(_name)).run()
    display(EKB_ART.table.round(4))
    for fig in render_report(EKB_ART, kind="gis", title="EKB NBCO").figures.values():
        display(fig)
else:
    print("[EKB NBCO] skipped (SUITE.run.ekb_nbco=False)")

# EKB alpha sweep

Run only the Ekaterinburg alpha sweep with adjustment target 0.3. Results are written to a separate table and route dump so existing paper results are not overwritten.


In [ ]:
# EKB alpha sweep: route/connectivity trade-off. The EKB NBCO run above already
# sweeps the alpha grid (the cfg sweep) -- reuse its table; run standalone if
# NBCO was skipped.
from IPython.display import display
from connectpt.routes_generator import ExperimentRunFactory, load_experiment

if SUITE.run.ekb_alpha_sweep:
    if "EKB_ART" in globals():
        EKB_SWEEP_DF = EKB_ART.table.round(4)
    else:
        _name = "ekb/case_study/our_nbco" + ("_smoke" if SUITE.smoke else "")
        EKB_SWEEP_DF = ExperimentRunFactory.from_cfg(load_experiment(_name)).run().table.round(4)
    display(EKB_SWEEP_DF)
else:
    print("[EKB sweep] skipped (SUITE.run.ekb_alpha_sweep=False)")

### EKB best solution (overwritten each run)

The best achieved EKB network (BCO incumbent) is written to fixed-name `artifacts/paper_results/final_ekb_best_solution.{csv,_routes.pt}` and overwritten on every run.

In [ ]:
# Save the single BEST achieved EKB solution (routes + row) to a FIXED stem,
# overwritten each run. Picks the min-cost row from the EKB NBCO sweep table.
EKB_BEST_STEM = "final_ekb_best_solution"
if "EKB_ART" not in globals() or EKB_ART.table.empty:
    print("[EKB best] skipped (EKB NBCO not run).")
else:
    _best = EKB_ART.table.loc[EKB_ART.table["cost"].idxmin()]
    _label, _alpha, _adj = _best["method"], _best["alpha"], _best["adj_target"]
    _key = f"{_label} a={_alpha} t={_adj}"
    _best_routes = as_route_tensor(EKB_ART.routes[_key])
    _best_row = EKB_ART.table[EKB_ART.table["alpha"] == _alpha].round(6)
    inst = EKB_ART.instance
    save_paper_table(_best_row, EKB_BEST_STEM, prefix=CTX.output_prefix)
    save_paper_routes(
        EKB_BEST_STEM,
        {"Initial EKB routes": as_route_tensor(inst.init_routes), _key: _best_routes},
        inst.coords, inst.street_adj,
        meta={"city": "EKB", "best_method": _key}, prefix=CTX.output_prefix)
    display(_best_row)
    print(f"[EKB best] overwritten -> {EKB_BEST_STEM}.csv + {EKB_BEST_STEM}_routes.pt (method={_key})")

In [ ]:
# Mumford0 neural-BCO variants (Our NBCO vs Trim 12 + extend 12, alpha=0.5) --
# top C-API. Config-first: cfg/experiments/m0/mumford0/batch[_smoke].yaml (2 method
# run configs); ExperimentBatch runs + concatenates the tables, render_report draws
# each run's route panels.
M0_NBCO_TABLE = ("final_mumford0_nbco_variants_alpha05_target03_iter200"
                 if SUITE.run.m0_nbco_variants else
                 "final_mumford0_nbco_variants_alpha05_target03_iter200_skipped")

if SUITE.run.m0_nbco_variants:
    from connectpt.routes_generator import ExperimentBatch, load_suite, render_report

    _name = "m0/mumford0/batch" + ("_smoke" if SUITE.smoke else "")
    batch = ExperimentBatch(load_suite(_name)).run()
    df = batch.table.drop(columns=["run"]).round(4)
    display(df)
    save_paper_table(df, M0_NBCO_TABLE, prefix=CTX.output_prefix)
    for art in batch.artifacts:
        rep = render_report(art, kind="network")
        for fig in rep.figures.values():
            display(fig)
else:
    print("Mumford0 NBCO variants skipped (suite.run.m0_nbco_variants=False)")

## E2 - Pareto fronts on RTT x WMC (Mumford0, covered_dup tier)

Both panels run on **Mumford0** from the **covered_dup** init tier (removable
redundancy), optimizing `alpha*RTT + (1-alpha)*WMC + adj(|.-target|)` with
`use_weighted_connectivity=True`. `alpha` = `route_time_weight` (so alpha=1 ->
pure RTT, alpha=0 -> pure WMC).

- **Fig 1 (our model):** RTT x WMC Pareto front of `GNN + trim/extend`, swept
  over `alpha` x `adj_target` (one curve per target).
- **Fig 2 (4-model comparison):** `{GNN, RPC} x {trim/extend, type2}`, alpha-
  swept at a fixed `adj_target=ADJ_TARGET` -> shows the trim/extend
  bee's contribution to the RTT x WMC front.

In [ ]:
# === E2 -- Pareto fronts on RTT x WMC (realistic init tier) ===
# Experiment design (city / alpha grid / adj targets / iterations) is config-first
# in cfg/experiments/e2/**. Only the city label -- read from the config -- is kept
# here, for output-file naming.
from connectpt.routes_generator import load_experiment
E2_CITY = load_experiment("e2/our_pareto/mumford1/our_nbco").data.city
print(f"[E2] city={E2_CITY} | run_our_pareto={SUITE.run.e2_our_pareto} run_5model={SUITE.run.e2_5model}")

In [ ]:
# E2 Our-NBCO Pareto sweep (alpha x adj_target) on Mumford1 -- top C-API.
# Config-first: cfg/experiments/e2/our_pareto/mumford1/our_nbco[_smoke].yaml (2D
# sweep). ExperimentRunFactory runs it; render_report builds table + Pareto figure.
e2_suffix = ("_smoke" if SUITE.smoke else "") + SUITE.e2.table_suffix

if SUITE.run.e2_our_pareto:
    from connectpt.routes_generator import (ExperimentRunFactory, load_experiment,
                                            render_report)

    _name = "e2/our_pareto/mumford1/our_nbco" + ("_smoke" if SUITE.smoke else "")
    art = ExperimentRunFactory.from_cfg(load_experiment(_name)).run()
    report = render_report(art)
    display(report.table)
    save_paper_table(art.table.round(4), f"final_e2_our_pareto_{E2_CITY}{e2_suffix}",
                     prefix=CTX.output_prefix)
    for fig in report.figures.values():
        display(fig)
else:
    print(f"[E2] {E2_CITY}: our-model Pareto sweep skipped (suite.run.e2_our_pareto=False)")

In [ ]:
# E2 Fig-2: 5-model RTT x WMC comparison on Mumford1, alpha-swept, adjustment OFF
# -- top C-API. Config-first: cfg/experiments/e2/5model/mumford1/batch[_smoke].yaml
# (5 method run configs); ExperimentBatch runs them + concatenates the tables.
e2_5model_table = f"final_e2_5model_{E2_CITY}{e2_suffix}"
if SUITE.run.e2_5model:
    from connectpt.routes_generator import ExperimentBatch, load_suite, render_report

    _name = "e2/5model/mumford1/batch" + ("_smoke" if SUITE.smoke else "")
    batch = ExperimentBatch(load_suite(_name)).run()
    E2_ABL_DF = batch.table.drop(columns=["run"]).round(4)   # method-labelled schema
    report = render_report(batch)                             # Pareto by method
    display(E2_ABL_DF)
    save_paper_table(E2_ABL_DF, e2_5model_table, prefix=CTX.output_prefix)
    for fig in report.figures.values():
        display(fig)
else:
    print(f"[E2] {E2_CITY}: 5-model alpha sweep skipped (suite.run.e2_5model=False)")
    E2_ABL_DF = pd.DataFrame()

### E2 Route Visualisation

Draw selected `alpha x adj_target` route sets from the E2 our-model sweep. The first grid shows the route sets directly; the second grid highlights changes against the LC initial network.

In [ ]:
# Visualise E2 our-model routes -- library reports.plot_routes_grid (plain grid +
# diff-vs-Initial grid). Reads the dump saved by the E2 pareto cell.
import torch
from connectpt.routes_generator.data import as_route_tensor
from connectpt.routes_generator.reports import plot_routes_grid
from connectpt.routes_generator.reports.paper_io import paper_path

if SUITE.run.e2_route_viz:
    e2_suffix = ("_smoke" if SUITE.smoke else "") + SUITE.e2.table_suffix
    dump_path = paper_path(f"final_e2_our_pareto_{E2_CITY}{e2_suffix}_routes.pt",
                           prefix=CTX.output_prefix)
    if not dump_path.exists():
        print(f"[E2 route viz] skipped: no dump {dump_path.name} (run the E2 our-pareto cell first)")
    else:
        dump = torch.load(dump_path, weights_only=False)
        route_sets = {label: as_route_tensor(rt) for label, rt in dump["routes"].items()}
        ref_label = next((m for m in route_sets if "Initial" in m), list(route_sets)[0])
        display(plot_routes_grid(route_sets, dump["coords"], dump["street_adj"],
                                 title=f"E2 {E2_CITY}: our-model routes"))
        display(plot_routes_grid(route_sets, dump["coords"], dump["street_adj"],
                                 diff_against=ref_label, title=f"E2 {E2_CITY}: diff vs {ref_label}"))
else:
    print("E2 route viz skipped (suite.run.e2_route_viz=False)")

## E1 narrow -- neural BCO vs Our NBCO alpha sweep

Runs only two methods on every benchmark city (`Mandl`, `Mumford0`, `Mumford1`, `Mumford2`, `Mumford3`):
`neural BCO` and `Our NBCO (GNN rebuild + trim/extend)`. For each method we run
`alpha in {0, 0.5, 1}` with `adjustment_degree_target=0.5` and 100 BCO iterations.

All SA / GA / HH / heuristic BCO / NSGA-II / trim-only branches are disabled here. Tables are saved as
`paper_results/final_main_unified_<city><E1U_TABLE_SUFFIX>.csv` and a combined table as
`final_main_unified_comparison<E1U_TABLE_SUFFIX>.csv`.


In [ ]:
# E1 narrow: neural BCO vs Our NBCO alpha sweep -- top C-API (ExperimentBatch).
# Config-first: cfg/experiments/e1/<city>/batch[_smoke].yaml lists the two method
# run configs (each its own alpha sweep); ExperimentBatch runs + concatenates them.
from connectpt.routes_generator import ExperimentBatch, load_suite, render_report

e1_suffix = ("_smoke" if SUITE.smoke else "") + SUITE.e1.table_suffix
all_rows = []
for city in SUITE.e1.cities:
    print(f"=== {city} E1 narrow: neural BCO + Our NBCO ===", flush=True)
    _name = f"e1/{city.lower()}/batch" + ("_smoke" if SUITE.smoke else "")
    batch = ExperimentBatch(load_suite(_name)).run()
    df = batch.table.drop(columns=["run"]).round(3)
    report = render_report(batch)
    display(df)
    save_paper_table(df, f"final_main_unified_{city}{e1_suffix}", prefix=CTX.output_prefix)
    all_rows += df.to_dict("records")

unified_df = pd.DataFrame(all_rows).round(3)
if not unified_df.empty:
    display(unified_df)
    save_paper_table(unified_df, "final_main_unified_comparison" + e1_suffix,
                     prefix=CTX.output_prefix)
else:
    print("E1 narrow skipped: no cities configured (suite.e1.cities is empty).")

### Route visualisation (unified run: RTT + WMC + adj for all methods)

Drawn from the **unified** dumps (`final_main_unified_<city>`). Layout: OD demand -> init -> methods (plain & diff vs init), plus a focused 1x4 (our routes / init / neural-BCO diff / our-NBCO diff).

In [ ]:
# Route-set visualisation from saved E1 route dumps -- library reports.plot_routes_grid
# (plain panels + a diff-vs-Initial grid, one style).
import torch
from connectpt.routes_generator.data import as_route_tensor
from connectpt.routes_generator.reports import plot_routes_grid
from connectpt.routes_generator.reports.paper_io import paper_path

VIZ_CITY = "Mumford0"
VIZ_NCOL = 3
suffix = ("_smoke" if SUITE.smoke else "") + SUITE.e1.table_suffix
dump_path = paper_path(f"final_main_unified_{VIZ_CITY}{suffix}_routes.pt",
                       prefix=CTX.output_prefix)
if not dump_path.exists():
    print(f"[route viz] skipped: no dump {dump_path.name} (run E1u for {VIZ_CITY} first)")
else:
    dump = torch.load(dump_path, weights_only=False)
    route_sets = {label: as_route_tensor(rt) for label, rt in dump["routes"].items()}
    coords, street_adj = dump["coords"], dump["street_adj"]
    ref_label = next((m for m in route_sets if "Initial" in m), list(route_sets)[0])
    display(plot_routes_grid(route_sets, coords, street_adj, ncols=VIZ_NCOL,
                             title=f"{VIZ_CITY}: route sets"))
    display(plot_routes_grid(route_sets, coords, street_adj, ncols=VIZ_NCOL,
                             diff_against=ref_label, title=f"{VIZ_CITY}: diff vs {ref_label}"))

## MACSA Table B -- paper routes + Our NBCO alpha sweep

This section replaces the old generic MACSA probe with the exact Mandl-8 Table-B workflow used for the paper figures. It reads the fixed route sets from `datasets/MACSA_data/mandl_8/routes_*.txt`, scores them with the same E1-style metric plumbing, then runs Our NBCO from the original network with `adjustment_degree_target = adj(MACSA, original)`.

Outputs:
- `final_macsa_mandl8_alpha_sweep_iter100.*`: Our NBCO alpha sweep, `alpha=0.0..1.0` step `0.1`, 100 BCO iterations per run.
- `final_macsa_mandl8_tableb.*`: Table-B paper methods plus the best Our NBCO sweep solution that beats MACSA on both RTT and WMC. The old cap-mode row is intentionally not included.


In [ ]:
# MACSA Table-B case study: thick helpers now live in the library
# (connectpt.routes_generator.paper_experiments.macsa). Configure the runtime
# values, then import the helpers + scenario constants by their bare names so the
# orchestration cells below read unchanged.
from connectpt.routes_generator.paper_experiments import macsa
from connectpt.routes_generator.citygraph_dataset import load_macsa_tensors

MC = macsa.configure(CTX, smoke=SUITE.smoke, seeds=SUITE.seeds)
from connectpt.routes_generator.paper_experiments.macsa import *  # noqa: F401,F403

print("MACSA Table-B config:", MC.summary())

In [29]:
# Score the fixed Mandl-8 Table-B routes from the MACSA paper.
if not SUITE.run.macsa:
    MACSA_TABLEB_DF = pd.DataFrame()
    MACSA_TABLEB_ROUTES = {}
    print("MACSA Table-B skipped (RUN_MACSA_EXPERIMENTS=False).")
else:
    MACSA_TENSORS = load_macsa_tensors(MACSA_SCENARIO_DIR)
    MACSA_COORDS = MACSA_TENSORS["node_locs"]
    MACSA_STREET_ADJ = MACSA_TENSORS["street_adj"]
    MACSA_DEMAND = MACSA_TENSORS["demand"]
    MACSA_RAW_ROUTES = {
        MACSA_METHOD_TITLE[key]: macsa_read_routes_0indexed(MACSA_SCENARIO_DIR / f"routes_{key}.txt")
        for key in MACSA_METHOD_ORDER
    }
    MACSA_SPEC = macsa_build_spec(MACSA_RAW_ROUTES, int(MACSA_COORDS.shape[0]))
    MACSA_TABLEB_ROUTES = {
        method: macsa_pad_routes(routes, MACSA_SPEC["n_routes"], MACSA_SPEC["max_route_len"])
        for method, routes in MACSA_RAW_ROUTES.items()
    }
    MACSA_SEED_ROUTES = MACSA_TABLEB_ROUTES[MACSA_REF_METHOD]
    MACSA_ADJ_TARGET_FROM_MACSA = float(adj_vs_init(MACSA_TABLEB_ROUTES["MACSA"], MACSA_SEED_ROUTES))

    MACSA_TABLEB_ROWS = []
    for method in MACSA_TABLEB_ROUTES:
        row, scored = macsa_score_routes(
            method, "macsa_table_b", MACSA_TABLEB_ROUTES[method],
            seed_routes=MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
            alpha=MACSA_TABLEB_EVAL_ALPHA,
            adj_target=MACSA_TABLEB_EVAL_ADJ_TARGET,
            adj_objective=MACSA_TABLEB_EVAL_ADJ_OBJECTIVE)
        row.update(alpha=np.nan, run_alpha=np.nan, n_iterations=np.nan,
                   macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA)
        MACSA_TABLEB_ROWS.append(row)
        MACSA_TABLEB_ROUTES[method] = scored
        print(f"  {method:20} ATT={row['ATT']:.2f} RTT={row['RTT']:.0f} "
              f"WMC={row['WMC']:.2f} adj={row['adj_vs_seed']:.3f} cost={row['cost']:.3f}")

    MACSA_TABLEB_DF = pd.DataFrame(MACSA_TABLEB_ROWS).round(6)
    save_paper_table(MACSA_TABLEB_DF, MACSA_ARTICLE_STEM, prefix=CTX.output_prefix)
    save_paper_routes(MACSA_ARTICLE_STEM, MACSA_TABLEB_ROUTES,
                      MACSA_COORDS, MACSA_STREET_ADJ,
                      meta={"scenario": MACSA_SCENARIO_NAME,
                            "ref_method": MACSA_REF_METHOD,
                            "eval_alpha": MACSA_TABLEB_EVAL_ALPHA,
                            "eval_adj_target": MACSA_TABLEB_EVAL_ADJ_TARGET,
                            "macsa_adj_target": MACSA_ADJ_TARGET_FROM_MACSA},
                      prefix=CTX.output_prefix)
    print(f"MACSA adj target from Table-B MACSA vs original: {MACSA_ADJ_TARGET_FROM_MACSA:.6f}")
    display(MACSA_TABLEB_DF)


NameError: name 'load_macsa_tensors' is not defined

In [ ]:
# Run Our NBCO alpha sweep: alpha=0.0..1.0 step 0.1, exactly 100 BCO iterations each.
if not SUITE.run.macsa or MACSA_TABLEB_DF.empty:
    MACSA_SWEEP_DF = pd.DataFrame()
    MACSA_SWEEP_ROUTES = {}
    print("MACSA alpha sweep skipped: no scored Table-B routes.")
else:
    sweep_csv = PAPER_DIR / f"{MC.sweep_stem}.csv"
    sweep_routes_path = PAPER_DIR / f"{MC.sweep_stem}_routes.pt"
    cached_routes = {}
    cached_duration = {}
    if sweep_routes_path.exists() and sweep_csv.exists() and not MACSA_SWEEP_FORCE_RERUN:
        try:
            payload = torch.load(sweep_routes_path, map_location="cpu", weights_only=False)
            meta = payload.get("meta", {})
            if int(meta.get("bco_iterations", -1)) == int(MC.bco_iterations):
                cached_routes = {k: as_route_tensor(v) for k, v in payload.get("routes", {}).items()}
                cache_df = pd.read_csv(sweep_csv)
                if "method" in cache_df:
                    cached_duration = dict(zip(cache_df["method"].astype(str),
                                               cache_df.get("duration_s", pd.Series(dtype=float))))
                print(f"[macsa sweep] loaded cache: {sweep_routes_path.name}")
            else:
                print("[macsa sweep] cache ignored: iteration count mismatch")
        except Exception as exc:
            print(f"[macsa sweep] cache ignored: {exc}")

    MACSA_SWEEP_ROWS = []
    MACSA_SWEEP_ROUTES = {}

    def macsa_save_sweep_progress():
        df = pd.DataFrame(MACSA_SWEEP_ROWS).sort_values("alpha").round(6)
        save_paper_table(df, MC.sweep_stem, prefix=CTX.output_prefix)
        save_paper_routes(MC.sweep_stem, MACSA_SWEEP_ROUTES,
                          MACSA_COORDS, MACSA_STREET_ADJ,
                          meta={"scenario": MACSA_SCENARIO_NAME,
                                "ref_method": MACSA_REF_METHOD,
                                "alpha_grid": MACSA_ALPHA_GRID,
                                "bco_iterations": MC.bco_iterations,
                                "bco_bees": MACSA_SWEEP_BEES,
                                "seed": MC.seed,
                                "adj_target": MACSA_ADJ_TARGET_FROM_MACSA,
                                "adj_objective": "target"},
                          prefix=CTX.output_prefix)
        return df

    for alpha in tqdm(MACSA_ALPHA_GRID, desc="MACSA Our NBCO alpha sweep"):
        label = macsa_alpha_label(alpha, MC.bco_iterations)
        if label in cached_routes and not MACSA_SWEEP_FORCE_RERUN:
            print(f"  [cache] {label}")
            routes = macsa_pad_routes(cached_routes[label], MACSA_SPEC["n_routes"], MACSA_SPEC["max_route_len"])
            duration = float(cached_duration.get(label, np.nan)) if label in cached_duration else np.nan
            source = "our_nbco_alpha_sweep_cached"
        else:
            print(f"  [run] {label}: alpha={alpha:.1f}, iters={MC.bco_iterations}", flush=True)
            routes, duration = macsa_run_our_nbco(
                MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
                alpha=float(alpha), adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                adj_objective="target", n_iterations=MC.bco_iterations,
                n_bees=MACSA_SWEEP_BEES, seed=MC.seed,
                force_cpu=MACSA_FORCE_CPU,
                edit_weights_path=MC.edit_weights_path,
                edit_adj_cond_feats=MC.edit_adj_cond_feats)
            source = "our_nbco_alpha_sweep"
        row, scored = macsa_score_routes(
            label, source, routes, seed_routes=MACSA_SEED_ROUTES,
            tensors=MACSA_TENSORS, spec=MACSA_SPEC,
            alpha=float(alpha), adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
            adj_objective="target")
        row.update(duration_s=duration, alpha=float(alpha), run_alpha=float(alpha),
                   n_iterations=int(MC.bco_iterations),
                   macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                   beats_macsa_rtt_wmc=bool(
                       (float(row["RTT"]) < float(MACSA_TABLEB_DF.loc[MACSA_TABLEB_DF["method"] == "MACSA", "RTT"].iloc[0])) and
                       (float(row["WMC"]) < float(MACSA_TABLEB_DF.loc[MACSA_TABLEB_DF["method"] == "MACSA", "WMC"].iloc[0]))))
        MACSA_SWEEP_ROWS = macsa_upsert_row(MACSA_SWEEP_ROWS, row)
        MACSA_SWEEP_ROUTES[label] = scored
        MACSA_SWEEP_DF = macsa_save_sweep_progress()
        print(f"    -> RTT={row['RTT']:.0f} WMC={row['WMC']:.2f} "
              f"ATT={row['ATT']:.2f} adj={row['adj_vs_seed']:.3f} cost={row['cost']:.3f} "
              f"duration={duration:.1f}s", flush=True)

    MACSA_SWEEP_DF = pd.DataFrame(MACSA_SWEEP_ROWS).sort_values("alpha").round(6)
    display(MACSA_SWEEP_DF)


In [ ]:
# Pick the best Our NBCO sweep solution that beats MACSA on both RTT and WMC,
# then build the comparison table: all Table-B methods + that single Our solution.
if not SUITE.run.macsa or MACSA_SWEEP_DF.empty:
    MACSA_COMPARISON_DF = pd.DataFrame()
    MACSA_COMPARISON_ROUTES = {}
    print("MACSA comparison skipped: no sweep rows.")
else:
    macsa_ref_row = MACSA_TABLEB_DF[MACSA_TABLEB_DF["method"] == "MACSA"].iloc[0]
    MACSA_BEST_SWEEP_ROW = macsa_select_best_sweep_row(MACSA_SWEEP_DF, macsa_ref_row)
    MACSA_BEST_SWEEP_LABEL = str(MACSA_BEST_SWEEP_ROW["method"])
    MACSA_BEST_ALPHA = float(MACSA_BEST_SWEEP_ROW["alpha"])
    MACSA_BEST_BEATS_MACSA = bool(MACSA_BEST_SWEEP_ROW["beats_macsa_rtt_wmc"])
    MACSA_BEST_COMPARE_LABEL = f"Our NBCO best (alpha={MACSA_BEST_ALPHA:.1f}, iter={MC.bco_iterations})"

    best_compare_row, best_compare_routes = macsa_score_routes(
        MACSA_BEST_COMPARE_LABEL, "our_nbco_alpha_sweep_best",
        MACSA_SWEEP_ROUTES[MACSA_BEST_SWEEP_LABEL],
        seed_routes=MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
        alpha=MACSA_TABLEB_EVAL_ALPHA,
        adj_target=MACSA_TABLEB_EVAL_ADJ_TARGET,
        adj_objective=MACSA_TABLEB_EVAL_ADJ_OBJECTIVE)
    best_compare_row.update(alpha=MACSA_BEST_ALPHA,
                            run_alpha=MACSA_BEST_ALPHA,
                            n_iterations=int(MC.bco_iterations),
                            macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                            beats_macsa_rtt_wmc=MACSA_BEST_BEATS_MACSA,
                            selected_from=MACSA_BEST_SWEEP_LABEL,
                            selection_rule="min RTT/MACSA_RTT + WMC/MACSA_WMC among rows beating MACSA on both")

    MACSA_COMPARISON_ROUTES = dict(MACSA_TABLEB_ROUTES)
    MACSA_COMPARISON_ROUTES[MACSA_BEST_COMPARE_LABEL] = best_compare_routes
    MACSA_COMPARISON_DF = pd.concat(
        [MACSA_TABLEB_DF, pd.DataFrame([best_compare_row])], ignore_index=True, sort=False).round(6)
    save_paper_table(MACSA_COMPARISON_DF, MACSA_COMPARISON_STEM, prefix=CTX.output_prefix)
    save_paper_routes(MACSA_COMPARISON_STEM, MACSA_COMPARISON_ROUTES,
                      MACSA_COORDS, MACSA_STREET_ADJ,
                      meta={"scenario": MACSA_SCENARIO_NAME,
                            "ref_method": MACSA_REF_METHOD,
                            "article_methods": [MACSA_METHOD_TITLE[k] for k in MACSA_METHOD_ORDER],
                            "best_our_method": MACSA_BEST_COMPARE_LABEL,
                            "best_sweep_method": MACSA_BEST_SWEEP_LABEL,
                            "best_beats_macsa_rtt_wmc": MACSA_BEST_BEATS_MACSA,
                            "bco_iterations": MC.bco_iterations,
                            "sweep_stem": MC.sweep_stem,
                            "cap_mode_included": False},
                      prefix=CTX.output_prefix)
    print("Best Our NBCO sweep solution:", {
        "method": MACSA_BEST_SWEEP_LABEL,
        "alpha": MACSA_BEST_ALPHA,
        "beats_macsa_rtt_wmc": MACSA_BEST_BEATS_MACSA,
        "RTT": float(MACSA_BEST_SWEEP_ROW["RTT"]),
        "WMC": float(MACSA_BEST_SWEEP_ROW["WMC"]),
        "ATT": float(MACSA_BEST_SWEEP_ROW["ATT"]),
        "adj": float(MACSA_BEST_SWEEP_ROW["adj_vs_seed"]),
    })
    display(MACSA_COMPARISON_DF)


In [ ]:
# Visualize only Our NBCO alpha-sweep solutions.
if not SUITE.run.macsa or MACSA_SWEEP_DF.empty:
    print("MACSA our-only visualization skipped: no sweep rows.")
else:
    sweep_order = [str(row["method"]) for _, row in MACSA_SWEEP_DF.sort_values("alpha").iterrows()]
    sweep_routes_ordered = {name: MACSA_SWEEP_ROUTES[name] for name in sweep_order}
    sweep_rows_by_method = {str(row["method"]): row.to_dict()
                            for _, row in MACSA_SWEEP_DF.iterrows()}

    fig = macsa_draw_grid(
        routes=sweep_routes_ordered, rows_by_method=sweep_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=False, ref_key=None, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=False, ncol=4,
        title="Mandl-8 MACSA: Our NBCO alpha sweep, plain route sets")
    MACSA_SWEEP_PLAIN_PATH = macsa_save_fig(fig, MC.sweep_stem, "viz_plain",
                                            prefix=CTX.output_prefix)

    fig = macsa_draw_grid(
        routes=sweep_routes_ordered, rows_by_method=sweep_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=True, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=False, ncol=4,
        title="Mandl-8 MACSA: Our NBCO alpha sweep, diff vs original")
    MACSA_SWEEP_DIFF_PATH = macsa_save_fig(fig, MC.sweep_stem, "viz_diff",
                                           prefix=CTX.output_prefix)
    macsa_display_image(MACSA_SWEEP_PLAIN_PATH)
    macsa_display_image(MACSA_SWEEP_DIFF_PATH)


In [ ]:
# Visualize all paper Table-B solutions plus the selected best Our NBCO solution.
if not SUITE.run.macsa or MACSA_COMPARISON_DF.empty:
    print("MACSA comparison visualization skipped: no comparison table.")
else:
    comparison_rows_by_method = {str(row["method"]): row.to_dict()
                                 for _, row in MACSA_COMPARISON_DF.iterrows()}
    fig = macsa_draw_grid(
        routes=MACSA_COMPARISON_ROUTES, rows_by_method=comparison_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=False, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=True, ncol=5,
        title="Mandl-8 MACSA Table B: paper methods + best Our NBCO")
    MACSA_COMPARISON_PLAIN_PATH = macsa_save_fig(fig, MACSA_COMPARISON_STEM, "viz_plain",
                                                 prefix=CTX.output_prefix)

    fig = macsa_draw_grid(
        routes=MACSA_COMPARISON_ROUTES, rows_by_method=comparison_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=True, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=True, ncol=5,
        title="Mandl-8 MACSA Table B: paper methods + best Our NBCO, diff vs original")
    MACSA_COMPARISON_DIFF_PATH = macsa_save_fig(fig, MACSA_COMPARISON_STEM, "viz_diff",
                                                prefix=CTX.output_prefix)
    macsa_display_image(MACSA_COMPARISON_PLAIN_PATH)
    macsa_display_image(MACSA_COMPARISON_DIFF_PATH)
